# Lesson 4.5: Technical Reference — Running the Full Geoparser


**🎬 Video:** [Lesson 4.5: Technical Reference — Running the Full Geoparser](#)

## Overview

This notebook documents how the pre-processed geoparsing output was produced for the project data. It is **not part of the assignment** and students are not expected to read or run it. It is provided for transparency and reproducibility.

---

The pipeline runs in five stages:

1. **Initialize** — Import libraries and load the geoparser with the SpaCy NER model, the geographic transformer, and the GeoNames gazetteer.
2. **Load** — Read the tokenized sentence data produced by Lesson 4.2 NER step.
3. **Parse** — Batch-process all sentences through the geoparser (~1 hour runtime).
4. **Filter & Explode** — Remove rows with no resolved toponyms; expand list columns to one row per toponym.
5. **Save** — Write output to `.pickle` and `.csv`.

> ⚠️ **Requirements:** This notebook requires `geoparser`, `en_core_web_trf`, `tqdm`, and the GeoNames database (~13 GB). These are **not installed** in the student Codespace environment. Run this only on a machine where the full stack has been set up.

---

## 1 Initialize

In [ ]:
from geoparser import Geoparser
from geoparser.recognizers import SpacyRecognizer
from geoparser.resolvers import SentenceTransformerResolver
from tqdm.notebook import tqdm
import pandas as pd
import warnings

warnings.simplefilter(action="ignore", category=FutureWarning)
print("✅ Libraries imported")

In [ ]:
print("Initializing geoparser... (this may take a minute)")
geo = Geoparser(
    recognizer=SpacyRecognizer("en_core_web_trf"),
    resolver=SentenceTransformerResolver("dguzh/geo-all-distilroberta-v1"),
)
print("✅ Geoparser initialized")

---

## 2 Load the Tokenized Sentence Data

The input is `JMU_toponyms.pickle` â€“ a DataFrame of tokenized sentences from the JMU Reddit data produced in Lesson 4.2. Each row is a sentence; the NER step has already flagged which sentences contain candidate place names.

In [ ]:
df_jmu_reddit_toponyms = pd.read_pickle("../data/JMU/JMU_toponyms.pickle")
print(f"✅ Loaded {len(df_jmu_reddit_toponyms):,} sentences")

---

## 3 Batch Geoparsing Function

`geoparse_dataframe()` wraps the geoparser's `.parse()` method to handle a full DataFrame. It passes all sentences as a batch (more efficient than sentence-by-sentence), extracts the resolved place name, coordinates, and GeoNames feature class for each toponym, and stores the results as list columns — one list per sentence.

In [ ]:
def geoparse_dataframe(df, text_column="sentences"):
    """
    Extract geographic locations from text data.
    
    Args:
        df: DataFrame with text data
        text_column: Column containing the text to parse
    
    Returns:
        DataFrame with added location columns
    """
    print(f"🔍 Processing {len(df)} sentences for geographic locations...")
    
    sentences = df[text_column].tolist()
    
    docs = geo.parse(sentences)
    
    places, latitudes, longitudes, feature_names = [], [], [], []
    
    for doc in tqdm(docs, desc="Extracting locations"):
        doc_places = []
        doc_latitudes = []
        doc_longitudes = []
        doc_feature_names = []
        
        for toponym in doc.toponyms:
            if toponym.location:
                doc_places.append(toponym.location.get("name"))
                doc_latitudes.append(toponym.location.get("latitude"))
                doc_longitudes.append(toponym.location.get("longitude"))
                doc_feature_names.append(toponym.location.get("feature_name"))
        
        places.append(doc_places)
        latitudes.append(doc_latitudes)
        longitudes.append(doc_longitudes)
        feature_names.append(doc_feature_names)
    
    df_result = df.copy()
    df_result["place"] = places
    df_result["latitude"] = latitudes
    df_result["longitude"] = longitudes
    df_result["feature_name"] = feature_names
    
    print(f"✅ Geoparsing complete")
    return df_result

---

## 4 Run, Filter, and Explode

1. **Run** — `geoparse_dataframe()` is called on the full dataset. Expect ~1 hour of runtime.
2. **Filter** — Rows where no toponyms were resolved (empty `place` lists) are removed.
3. **Explode** — List columns (`place`, `latitude`, `longitude`, `feature_name`) are expanded to one row per toponym, enabling grouping and mapping.

In [ ]:
# Run the geoparser over the entire 'sentences' column (~1 hour)
geoparse_results = geoparse_dataframe(df_jmu_reddit_toponyms)

In [ ]:
# Remove rows where no locations were resolved
df_reddit_geoparsed = geoparse_results[geoparse_results["place"].apply(len) != 0]
print(f"Rows with at least one resolved location: {len(df_reddit_geoparsed):,}")

In [ ]:
# Explode list columns to one row per toponym
df_reddit_geoparsed_long = df_reddit_geoparsed.explode(
    ["place", "latitude", "longitude", "feature_name"]
).reset_index(drop=True)
df_reddit_geoparsed_long.sample(10)

---

## 5 Save Output

Output is written to both `.pickle` (for downstream Python use) and `.csv` (for spreadsheet review and manual correction).

In [ ]:
df_reddit_geoparsed_long.to_pickle("../data/JMU/JMU_geoparsed_long.pickle")
df_reddit_geoparsed_long.to_csv("../data/JMU/JMU_geoparsed_long.csv", index=False)
print(f"✅ Saved {len(df_reddit_geoparsed_long):,} rows")

---

## Lesson Summary

### Part 1: Initialize
- Loads the geoparser library and configures it for batch processing

### Part 2: Load the Tokenized Sentence Data
- `pd.read_pickle('file.pickle')` — reloads the sentence-level DataFrame produced in Lessons 4.1–4.2

### Part 3: Batch Geoparsing Function
- `geo.parse(text)` — the core call; wrapped in a loop with error handling so a single failed parse does not abort the full run
- Results are accumulated in plain Python lists before being assembled into a DataFrame

### Part 4: Run, Filter, and Explode
- The batch function is applied to every sentence; results are filtered to keep only the highest-confidence candidate per place name
- `df.explode('col', ignore_index=True)` — flattens the list of parsed places into one row per place per sentence

### Part 5: Save Output
- `df.sample(n)` — spot-checks the final output before saving
- Saves to both CSV and pickle so downstream notebooks can load whichever format they need

> 👉 **Note:** *This notebook is an instructor reference — it does not need to be run by students. The output file it produces (`jmu_reddit_geoparsed_long.pickle`) is already saved in the `data/JMU/` folder. Return to [Lesson 4.3](lesson_4_3_geoparsing_mapping.ipynb) to continue the lesson sequence.*

---

➡️ **Next:** [Lesson 5.1 — Sentiment Analysis](../lesson_5_sentiment_analysis/lesson_5_1_sentiment_analysis.ipynb)